# Day 21 — Best-Candidate Checkpoint and Week-4 Direction

## Objective

The objective of Day 21 is to consolidate the experimental results obtained so far, compare baseline and tuned candidates, review representative successes and failures, document remaining weaknesses, and define the proposed Week-4 final direction for mentor review.

## Candidates Reviewed

### Anomaly Detection
- Statistical Z-score baseline
- Isolation Forest baseline
- Tuned Isolation Forest

### Forecasting
- Persistence baseline
- Random Forest baseline
- Tuned Random Forest

## Evaluation Policy

- Training data was used for model fitting.
- Validation data was used for tuning and candidate selection.
- The final test set was kept separate from the tuning process.
- Anomaly ground-truth labels are unavailable, so anomaly-rate behaviour and validation stability are used instead of precision, recall and F1.

## Checkpoint Outputs

1. Baseline versus tuned comparison
2. Project-specific feature ablation findings
3. Representative success and failure cases
4. Remaining weaknesses
5. Best-candidate checkpoint
6. Proposed Week-4 final direction
7. Mentor approval record

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

project_path = r"C:\Users\MESHWA\Downloads\underwater-water-quality-ai-analytics"

experiments_path = project_path + r"\experiments"
models_path = project_path + r"\models"

# Load Day 19 experiment log
experiment_log = pd.read_csv(
    experiments_path + r"\day19_experiment_log.csv"
)

print("Experiment log shape:", experiment_log.shape)
experiment_log.head()

Experiment log shape: (63, 14)


,Experiment ID,Model,n_estimators,contamination,max_samples,Train Anomaly Rate (%),Validation Anomaly Rate (%),Rate Difference (%),Processing Time (sec),max_depth,min_samples_split,min_samples_leaf,Validation MAE,Validation RMSE
0,IF_01,Isolation Forest,100,0.03,512.0,3.01,3.05,0.04,0.4603,NaN,NaN,NaN,NaN,NaN
1,IF_02,Isolation Forest,100,0.03,0.8,3.01,3.26,0.25,0.4204,NaN,NaN,NaN,NaN,NaN
2,IF_03,Isolation Forest,100,0.03,1.0,3.01,3.05,0.04,0.4378,NaN,NaN,NaN,NaN,NaN
3,IF_04,Isolation Forest,100,0.05,512.0,5.02,5.91,0.89,0.3846,NaN,NaN,NaN,NaN,NaN
4,IF_05,Isolation Forest,100,0.05,0.8,5.02,6.31,1.29,0.3823,NaN,NaN,NaN,NaN,NaN


In [2]:
checkpoint_comparison = pd.DataFrame({
    "Method": [
        "Z-score Statistical Baseline",
        "Isolation Forest Baseline",
        "Tuned Isolation Forest",
        "Persistence Forecast",
        "Random Forest Baseline",
        "Tuned Random Forest"
    ],
    "Task": [
        "Anomaly Detection",
        "Anomaly Detection",
        "Anomaly Detection",
        "Forecasting",
        "Forecasting",
        "Forecasting"
    ],
    "Selection / Evaluation": [
        "Statistical baseline",
        "Baseline configuration",
        "Day 19 validation tuning",
        "Persistence baseline",
        "Day 16 baseline",
        "Day 19 validation tuning"
    ],
    "Evidence": [
        "Anomaly-rate behaviour",
        "5% contamination baseline",
        "Validation stability",
        "MAE 1.1335 | RMSE 1.6095",
        "MAE 0.9816 | RMSE 1.2453",
        "Selected using validation MAE"
    ],
    "Status": [
        "Baseline",
        "Baseline",
        "Tuned candidate",
        "Baseline",
        "Candidate",
        "Tuned candidate"
    ]
})

checkpoint_comparison

,Method,Task,Selection / Evaluation,Evidence,Status
0,Z-score Statistical Baseline,Anomaly Detection,Statistical baseline,Anomaly-rate behaviour,Baseline
1,Isolation Forest Baseline,Anomaly Detection,Baseline configuration,5% contamination baseline,Baseline
2,Tuned Isolation Forest,Anomaly Detection,Day 19 validation tuning,Validation stability,Tuned candidate
3,Persistence Forecast,Forecasting,Persistence baseline,MAE 1.1335 | RMSE 1.6095,Baseline
4,Random Forest Baseline,Forecasting,Day 16 baseline,MAE 0.9816 | RMSE 1.2453,Candidate
5,Tuned Random Forest,Forecasting,Day 19 validation tuning,Selected using validation MAE,Tuned candidate


In [3]:
forecast_checkpoint = pd.DataFrame({
    "Method": [
        "Persistence Baseline",
        "Random Forest Baseline",
        "Tuned Random Forest"
    ],
    "MAE": [
        1.133537,
        0.981553,
        np.nan
    ],
    "RMSE": [
        1.609520,
        1.245266,
        np.nan
    ],
    "Evaluation Stage": [
        "Day 16 test evaluation",
        "Day 16 test evaluation",
        "Day 19 validation selection"
    ]
})

forecast_checkpoint

,Method,MAE,RMSE,Evaluation Stage
0,Persistence Baseline,1.133537,1.609520,Day 16 test evaluation
1,Random Forest Baseline,0.981553,1.245266,Day 16 test evaluation
2,Tuned Random Forest,NaN,NaN,Day 19 validation selection


In [4]:
ablation_path = (
    experiments_path +
    r"\day20_feature_ablation_results.csv"
)

ablation_df = pd.read_csv(ablation_path)

ablation_df

,Removed Feature,Remaining Features,Train Anomaly Rate (%),Validation Anomaly Rate (%),Validation Rate Change (%),Absolute Change (%)
0,ph,8,3.01,2.04,-1.02,1.02
1,Hardness,8,3.01,3.05,0.00,0.00
2,Solids,8,3.01,2.85,-0.20,0.20
3,Chloramines,8,3.01,1.63,-1.43,1.43
4,Sulfate,8,3.01,2.65,-0.41,0.41
5,Conductivity,8,3.01,2.24,-0.81,0.81
6,Organic_carbon,8,3.01,2.24,-0.81,0.81
7,Trihalomethanes,8,3.01,3.05,0.00,0.00
8,Turbidity,8,3.01,2.44,-0.61,0.61


## Representative Successes

### Anomaly Detection
The Isolation Forest pipeline successfully produces anomaly flags from the available water-quality variables without requiring labelled anomaly data.

### Forecasting
The Random Forest forecasting baseline achieved lower test MAE and RMSE than the persistence baseline in the Day 16 evaluation.

### Robustness Analysis
Feature ablation quantified how removing individual water-quality variables changes anomaly-detection behaviour.

### Monitoring
The monitoring visualizations connect anomaly flags with time-stamped sensor measurements, providing an interpretable view of unusual observations.

## Representative Failures / Unusual Behaviour

### Anomaly Detection
Without ground-truth anomaly labels, detected anomalies cannot be confirmed as true environmental events.

### Forecasting
The forecasting model may perform differently across chronological periods and may struggle with sudden or unusual changes.

### Temporal Limitation
The timestamps were synthetically generated because the original dataset did not contain real temporal measurements.

### Dataset Limitation
The available dataset does not contain temperature, dissolved oxygen, or pressure, so the system cannot currently evaluate those parameters.

### Distribution Limitation
Extreme or unusual sensor values may be flagged even when they represent legitimate environmental conditions.

## Remaining Weaknesses

1. No ground-truth anomaly labels are available.
2. Timestamps are synthetic rather than real sensor timestamps.
3. The dataset is relatively small.
4. Important underwater variables such as temperature, dissolved oxygen and pressure are unavailable.
5. Current forecasting focuses on next-step pH prediction.
6. Anomaly-rate evaluation cannot establish whether detected events are environmentally meaningful.
7. Model behaviour may change when applied to real sensor data with different sampling intervals or distributions.

# Best-Candidate Checkpoint

## Anomaly Detection Candidate

**Tuned Isolation Forest**

Reason for checkpointing:
- It was selected through systematic Day 19 tuning.
- Multiple hyperparameter configurations were evaluated.
- Training and validation anomaly behaviour was logged.
- Feature ablation and monitoring analysis were subsequently performed.

## Forecasting Candidate

**Tuned Random Forest Regressor**

Reason for checkpointing:
- It was selected through systematic Day 19 hyperparameter tuning.
- Validation MAE was used for configuration selection.
- Chronological validation was performed during Day 20.
- The final test set remains untouched for the tuned configuration.

## Checkpoint Decision

The two candidates will proceed to the final evaluation and reporting stage:

1. Tuned Isolation Forest for anomaly detection
2. Tuned Random Forest for next-step pH forecasting

# Proposed Week-4 Final Direction

## Final System

The proposed Week-4 system will combine two complementary components:

### 1. Water-Quality Anomaly Detection
Tuned Isolation Forest will generate anomaly flags and anomaly scores from the available water-quality variables.

### 2. Water-Quality Forecasting
Tuned Random Forest will predict the next pH value using current sensor measurements, temporal features and lag/rolling features.

## Final Evaluation

The final evaluation will report:

- Anomaly-detection behaviour
- Forecasting MAE
- Forecasting RMSE
- Chronological validation behaviour
- Feature-ablation findings
- Representative monitoring examples
- Processing considerations

## Final Reporting

The final report will clearly distinguish:

- Measured results
- Model-generated outputs
- Dataset limitations
- Synthetic timestamp assumptions
- Absence of ground-truth anomaly labels

## Mentor Approval Required

The Week-4 direction should be reviewed with the mentor before proceeding to the final implementation and evaluation stage.

# Mentor Review Record

## Proposed Week-4 Direction

- Tuned Isolation Forest → anomaly detection
- Tuned Random Forest → next-step pH forecasting
- Chronological validation → robustness evidence
- Feature ablation → project-specific analysis
- Monitoring visualization → interpretable presentation
- Final test evaluation → performed after checkpoint approval

## Mentor Feedback

**Date:** __________________

**Mentor:** __________________

**Feedback / Corrections:**

__________________________________________________

__________________________________________________

__________________________________________________

## Approval Status

- [ ] Approved
- [ ] Approved with modifications
- [ ] Further experiments requested

## Required Changes

__________________________________________________

__________________________________________________